#  Tầng 2c: Free Text Tagging

**Mục tiêu:** Xử lý file `raw/open_ended_responses.txt`, chuẩn hóa text  
(bỏ dấu câu, tách từ, gom nhóm theo câu hỏi).

**Input:** `raw/open_ended_responses.txt`  
**Output:** `processed/tagged_free_text.csv`

In [1]:
import pandas as pd
import re
import string
from pathlib import Path
from collections import Counter

from pathlib import Path
# Ensure paths resolve from the project root when notebooks run from notebooks/.
import os
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
# ── Config ──────────────────────────────────────────────
INPUT_FILE = "raw/open_ended_responses.txt"
OUTPUT_FILE = "processed/tagged_free_text.csv"

assert Path(INPUT_FILE).exists(), f"[ERROR] Không tìm thấy {INPUT_FILE}!"

raw_text = Path(INPUT_FILE).read_text(encoding="utf-8")
print(f"[OK] Đọc {INPUT_FILE}: {len(raw_text)} ký tự")

[OK] Đọc raw/open_ended_responses.txt: 3707 ký tự


In [2]:
# ══════════════════════════════════════════════════════════
# BƯỚC 1: PARSE - Tách theo câu hỏi và các câu trả lời
# ══════════════════════════════════════════════════════════
# File format:
#   Câu hỏi?:
#     1. Trả lời 1
#     2. Trả lời 2

def parse_open_ended(text):
    """
    Parse file text thành list of dict:
    [{question, response_id, response_text}, ...]
    """
    records = []
    current_question = None
    
    for line in text.strip().split("\n"):
        line = line.strip()
        if not line:
            continue
        
        # Nhận diện câu hỏi: kết thúc bằng "?:" hoặc ":" và không bắt đầu bằng số
        if line.endswith("?:") or (line.endswith(":") and not re.match(r"^\s*\d+\.", line)):
            current_question = line.rstrip(":")
            continue
        
        # Nhận diện câu trả lời: bắt đầu bằng "số."
        match = re.match(r"^(\d+)\.\s*(.+)", line)
        if match and current_question:
            resp_id = int(match.group(1))
            resp_text = match.group(2).strip()
            records.append({
                "question": current_question,
                "response_id": resp_id,
                "response_raw": resp_text,
            })
    
    return pd.DataFrame(records)

df_text = parse_open_ended(raw_text)
print(f"[OK] Parsed: {len(df_text)} câu trả lời từ {df_text['question'].nunique()} câu hỏi")
print(f"\n Các câu hỏi:")
for q in df_text["question"].unique():
    n = (df_text["question"] == q).sum()
    print(f"   • ({n} responses) {q}")

[OK] Parsed: 47 câu trả lời từ 6 câu hỏi

 Các câu hỏi:
   • (14 responses) Theo bạn, điều gì cần cải thiện nhất ở các công cụ CI/CD hiện nay tại Việt Nam?
   • (13 responses) Bạn có đề xuất nào để thúc đẩy áp dụng CI/CD hiệu quả hơn?
   • (12 responses) Những trang bị mà sinh viên cần để áp dụng CI/CD hiệu quả hơn là gì?
   • (3 responses) Bạn có dự định học hoặc áp dụng CI/CD trong thời gian tới không?
   • (3 responses) Theo bạn, rào cản lớn nhất khiến sinh viên chưa áp dụng CI/CD là gì?
   • (2 responses) Bạn mong đợi điều gì nhất nếu CI/CD được đưa vào học tập/dự án thực tế?


In [3]:
# ══════════════════════════════════════════════════════════
# BƯỚC 2: TEXT NORMALIZATION - Chuẩn hóa text
# ══════════════════════════════════════════════════════════

# Vietnamese stopwords (phổ biến)
VIETNAMESE_STOPWORDS = {
    "và", "hoặc", "là", "của", "cho", "với", "trong", "về",
    "để", "được", "có", "không", "các", "một", "những", "này",
    "đó", "khi", "nếu", "thì", "mà", "vì", "do", "từ",
    "bạn", "tôi", "nào", "như", "theo", "cần", "phải",
    "đã", "sẽ", "đang", "rất", "quá", "cũng", "vẫn",
    "hơn", "nhất", "rồi", "lại", "ra", "lên", "vào",
    "tại", "trên", "dưới", "nên", "thêm", "chỉ", "còn",
    "hay", "nhưng", "mới", "nhiều", "ít", "ai", "gì",
    "ko", "k", "vs",
}

VIETNAMESE_COMPOUND_WORDS = {
    # ── IT & DevOps Chuyên Ngành ───────────────────────
    "sinh viên": "sinh_viên",
    "giảng viên": "giảng_viên",
    "nhà trường": "nhà_trường",
    "doanh nghiệp": "doanh_nghiệp",
    "môn học": "môn_học",
    "học phần": "học_phần",
    "chương trình": "chương_trình",
    "giảng dạy": "giảng_dạy",
    "đào tạo": "đào_tạo",
    "đại học": "đại_học",
    "thực tập": "thực_tập",
    "lý thuyết": "lý_thuyết",
    "thực hành": "thực_hành",
    "thực tế": "thực_tế",
    "kinh nghiệm": "kinh_nghiệm",
    "kiến thức": "kiến_thức",
    "kỹ năng": "kỹ_năng",
    "công cụ": "công_cụ",
    "phần mềm": "phần_mềm",
    "mã nguồn": "mã_nguồn",
    "quy trình": "quy_trình",
    "sơ đồ": "sơ_đồ",
    "luồng đi": "luồng_đi",
    # ── Hoạt động & Triển khai ─────────────────────────
    "phát triển phần mềm": "phát_triển_phần_mềm",
    "phát triển": "phát_triển",
    "triển khai": "triển_khai",
    "áp dụng": "áp_dụng",
    "sử dụng": "sử_dụng",
    "tích hợp liên tục": "tích_hợp_liên_tục",
    "tích hợp": "tích_hợp",
    "tự động hóa": "tự_động_hóa",
    "kiểm thử tự động": "kiểm_thử_tự_động",
    "kiểm thử": "kiểm_thử",
    "tự động": "tự_động",
    "phát hiện lỗi": "phát_hiện_lỗi",
    "thiết lập": "thiết_lập",
    "cấu hình": "cấu_hình",
    "môi trường": "môi_trường",
    # ── Động từ / Danh từ khảo sát khác ───────────────
    "giới thiệu": "giới_thiệu",
    "thực hiện": "thực_hiện",
    "tìm hiểu": "tìm_hiểu",
    "tự tìm hiểu": "tự_tìm_hiểu",
    "tự học": "tự_học",
    "đề xuất": "đề_xuất",
    "thúc đẩy": "thúc_đẩy",
    "mong đợi": "mong_đợi",
    "hiệu quả": "hiệu_quả",
    "thông tin": "thông_tin",
    "nền tảng": "nền_tảng",
    "cung cấp": "cung_cấp",
    "lập trình": "lập_trình",
    "quản lý": "quản_lý",
    "tổ chức": "tổ_chức",
    "khó khăn": "khó_khăn",
    "rào cản": "rào_cản",
    "tiết kiệm": "tiết_kiệm",
    "thời gian": "thời_gian",
    # ── Khắc phục các âm tiết rời ở 3 biểu đồ mới ──────
    "cơ hội": "cơ_hội",
    "giảm bớt": "giảm_bớt",
    "công việc": "công_việc",
    "thủ công": "thủ_công",
    "lặp đi lặp lại": "lặp_đi_lặp_lại",
    "quá trình": "quá_trình",
    "tốc độ": "tốc_độ",
    "dự án": "dự_án",
    "đơn giản": "đơn_giản",
    "kế hoạch": "kế_hoạch",
    "thời gian tới": "thời_gian_tới",
    "chưa có": "chưa_có",
    "chưa áp dụng": "chưa_áp_dụng",
    "thiếu kiến thức": "thiếu_kiến_thức",
    "thiếu môi trường": "thiếu_môi_trường",
    "phổ biến": "phổ_biến",
    "đầy đủ": "đầy_đủ",
    "tiếp cận": "tiếp_cận",
    "biện pháp": "biện_pháp",
    "chủ động": "chủ_động"
}

def normalize_text(text):
    """Chuẩn hóa: lowercase, bỏ dấu câu, bỏ số đứng riêng, ghép từ ghép tiếng Việt."""
    if not isinstance(text, str):
        return ""
    text = text.lower().strip()
    # Bỏ dấu câu (giữ chữ cái Unicode + khoảng trắng)
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    # Bỏ số đứng riêng
    text = re.sub(r"\b\d+\b", "", text)
    # Ghép từ ghép tiếng Việt (ưu tiên cụm dài trước)
    for k in sorted(VIETNAMESE_COMPOUND_WORDS.keys(), key=len, reverse=True):
        text = text.replace(k, VIETNAMESE_COMPOUND_WORDS[k])
    # Chuẩn hóa khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_and_filter(text, stopwords=VIETNAMESE_STOPWORDS):
    """Tách từ và loại bỏ stopwords."""
    tokens = text.split()
    return [t for t in tokens if t not in stopwords and len(t) > 1]

# Áp dụng vectorized
df_text["response_clean"] = df_text["response_raw"].apply(normalize_text)
df_text["tokens"] = df_text["response_clean"].apply(tokenize_and_filter)
df_text["token_count"] = df_text["tokens"].str.len()

print(f"[OK] Đã chuẩn hóa text")
print(f"\n Token count stats:")
print(df_text["token_count"].describe().to_string())

[OK] Đã chuẩn hóa text

 Token count stats:
count    47.000000
mean      7.489362
std      11.507790
min       0.000000
25%       1.000000
50%       4.000000
75%       8.500000
max      49.000000


In [4]:
# ══════════════════════════════════════════════════════════
# BƯỚC 3: THỐNG KÊ TỪ KHÓA TOP theo từng câu hỏi
# ══════════════════════════════════════════════════════════

for question in df_text["question"].unique():
    mask = df_text["question"] == question
    all_tokens = df_text.loc[mask, "tokens"].explode()
    top_tokens = all_tokens.value_counts().head(10)
    
    print(f"\n Top 10 từ khóa: {question[:60]}...")
    for word, count in top_tokens.items():
        bar = "█" * count
        print(f"   {word:<20} {count:>3} {bar}")


 Top 10 từ khóa: Theo bạn, điều gì cần cải thiện nhất ở các công cụ CI/CD hiệ...
   sinh_viên              6 ██████
   ci                     5 █████
   cd                     5 █████
   học                    3 ███
   luồng                  3 ███
   phổ_biến               2 ██
   thực_hành              2 ██
   pipeline               2 ██
   dạy                    2 ██
   nhà_trường             2 ██

 Top 10 từ khóa: Bạn có đề xuất nào để thúc đẩy áp dụng CI/CD hiệu quả hơn?...
   sinh_viên              5 █████
   ci                     4 ████
   cd                     4 ████
   dẫn                    3 ███
   hướng                  3 ███
   hiểu                   3 ███
   quy_trình              3 ███
   giảng_viên             2 ██
   thể                    2 ██
   tổ_chức                2 ██

 Top 10 từ khóa: Những trang bị mà sinh viên cần để áp dụng CI/CD hiệu quả hơ...
   cd                     3 ███
   ci                     3 ███
   docker                 3 ███
   git           

In [5]:
# ══════════════════════════════════════════════════════════
# BƯỚC 4: TAGGING - Gán nhãn chủ đề tự động
# ══════════════════════════════════════════════════════════

# Từ điển nhãn dựa trên từ khóa
TAG_KEYWORDS = {
    "thực hành":       ["thực hành", "thực tế", "hands-on", "practice", "lab", "workshop"],
    "chương trình học": ["môn học", "chương trình", "đào tạo", "giảng dạy", "trường", "nhà trường", "giảng viên"],
    "tài liệu":        ["tài liệu", "hướng dẫn", "tutorial", "documentation"],
    "công cụ":         ["github", "gitlab", "jenkins", "docker", "git", "pipeline", "actions"],
    "kiến thức":       ["kiến thức", "kỹ năng", "hiểu", "học", "tiếp cận"],
    "cộng đồng":       ["cộng đồng", "seminar", "workshop", "trao đổi"],
    "dự án":           ["dự án", "project", "sản phẩm", "ứng dụng"],
    "tự động hóa":     ["tự động", "automation", "pipeline", "deploy", "build", "test"],
}

def auto_tag(text):
    """Gán nhãn cho 1 response dựa trên từ khóa."""
    text_lower = text.lower()
    tags = []
    for tag, keywords in TAG_KEYWORDS.items():
        if any(kw in text_lower for kw in keywords):
            tags.append(tag)
    return ", ".join(tags) if tags else "khác"

# Áp dụng vectorized
df_text["tags"] = df_text["response_raw"].apply(auto_tag)

# Thống kê tags
all_tags = df_text["tags"].str.split(", ").explode()
tag_counts = all_tags.value_counts()

print(f"  Phân bố Tags:")
for tag, count in tag_counts.items():
    bar = "█" * count
    print(f"   {tag:<18} {count:>3} {bar}")

  Phân bố Tags:
   khác                24 ████████████████████████
   kiến thức           16 ████████████████
   chương trình học    11 ███████████
   tự động hóa         10 ██████████
   thực hành            7 ███████
   công cụ              7 ███████
   dự án                6 ██████
   tài liệu             3 ███
   cộng đồng            2 ██


In [6]:
# ══════════════════════════════════════════════════════════
# BƯỚC 5: PREVIEW & LƯU FILE
# ══════════════════════════════════════════════════════════

# Chuẩn bị output DataFrame (chuyển tokens list → string để lưu CSV)
df_output = df_text.copy()
df_output["tokens"] = df_output["tokens"].apply(lambda x: " | ".join(x))

# Preview
print(" Preview:")
df_output[["question", "response_raw", "response_clean", "tags"]].head(10)

 Preview:


,question,response_raw,response_clean,tags
0,"Theo bạn, điều gì cần cải thiện nhất ở các côn...",Giới thiệu rộng rãi hơn,giới_thiệu rộng rãi hơn,khác
1,"Theo bạn, điều gì cần cải thiện nhất ở các côn...",Thêm môn học,thêm môn_học,"chương trình học, kiến thức"
2,"Theo bạn, điều gì cần cải thiện nhất ở các côn...",Ko,ko,khác
3,"Theo bạn, điều gì cần cải thiện nhất ở các côn...",Phổ biến đến các bạn sinh viên từ năm nhất,phổ_biến đến các bạn sinh_viên từ năm nhất,khác
4,"Theo bạn, điều gì cần cải thiện nhất ở các côn...",ko,ko,khác
5,"Theo bạn, điều gì cần cải thiện nhất ở các côn...","Để sinh viên học CI/CD hiệu quả hơn, việc giản...",để sinh_viên học ci cd hiệu_quả hơn việc giảng...,"thực hành, chương trình học, công cụ, kiến thứ..."
6,"Theo bạn, điều gì cần cải thiện nhất ở các côn...",Sinh viên khó hình dung luồng đi của code. Cần...,sinh_viên khó hình dung luồng_đi của code cần ...,"công cụ, tự động hóa"
7,"Theo bạn, điều gì cần cải thiện nhất ở các côn...",Các chương trình học chính nên dạy những kiến ...,các chương_trình học chính nên dạy những kiến_...,"chương trình học, kiến thức"
8,"Theo bạn, điều gì cần cải thiện nhất ở các côn...",dsfdsfds,dsfdsfds,khác
9,"Theo bạn, điều gì cần cải thiện nhất ở các côn...",nhi,nhi,khác


In [7]:
# Lưu
df_output.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

# Verify
verify = pd.read_csv(OUTPUT_FILE, encoding="utf-8-sig")
assert verify.shape[0] == len(df_text)
print(f"[OK] Đã lưu: {OUTPUT_FILE}")
print(f"   Shape: {verify.shape[0]} dòng × {verify.shape[1]} cột")
print(f"   Câu hỏi: {verify['question'].nunique()}")
print(f"   Tags unique: {verify['tags'].nunique()}")

[OK] Đã lưu: processed/tagged_free_text.csv
   Shape: 47 dòng × 7 cột
   Câu hỏi: 6
   Tags unique: 17
